### Fase 1: Transformando Canciones en "Canastas"
El algoritmo Apriori no lee DataFrames normales; necesita una lista de transacciones. 
En nuestro caso, cada **Canción** es una transacción, y los **Artistas** que participan en ella son los artículos de la canasta.

In [ ]:
# Instalar si es necesario: !pip install mlxtend
import re
import pandas as pd
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, association_rules

# 1. INGESTA DE DATOS
url = "https://raw.githubusercontent.com/mricardo89/data-mining/main/Unit-III/datasets/Most%20Streamed%20Spotify%20Songs%202024.csv"
df_spotify = pd.read_csv(url, encoding='latin-1')

# 2. LIMPIEZA: Eliminar nulos de la columna de artistas
df_spotify = df_spotify.dropna(subset=['Artist'])

# 3. TRANSFORMACIÓN: Separar los artistas que colaboran en cada canción
# La columna 'Artist' solo trae al artista principal, los invitados vienen en el
# nombre de la canción (ej. "Fortnight (feat. Post Malone)", "STAY (with Justin Bieber)")
# Convertimos esa información en una verdadera lista de Python

def split_artist(fila):
    # "Tyler, The Creator" es un solo artista, se protege para que no lo parta la coma
    artistas = [fila['Artist'].replace('Tyler, The Creator', 'Tyler The Creator')]
    invitados = re.search(r'[\(\[](?:feat\.?|ft\.?|with)\s+([^\)\]]+)[\)\]]', str(fila['Track']), flags=re.IGNORECASE)
    if invitados:
        artistas += re.split(r',|&|\band\b', invitados.group(1), flags=re.IGNORECASE)
    return list({a.strip() for a in artistas if a.strip()})

lista_colaboraciones = df_spotify.apply(split_artist, axis=1).tolist()

print("Muestra de las primeras 5 colaboraciones:")
for i, artistas in enumerate(lista_colaboraciones[:5], start=1):
    print(f"Canción {i}: {artistas}")


Muestra de las primeras 5 colaboraciones:
Canción 1: ['Tommy Richman']
Canción 2: ['Kendrick Lamar']
Canción 3: ['Artemas']
Canción 4: ['Miley Cyrus']
Canción 5: ['Eminem']


In [9]:

# Utiliza TransactionEncoder para convertir 'lista_colaboraciones' en una matriz booleana (One-Hot)
# Guárdala en un DataFrame llamado 'df_transacciones'

te = TransactionEncoder()
matriz_bool = te.fit_transform(lista_colaboraciones)
df_transacciones = pd.DataFrame(matriz_bool, columns=te.columns_)

print("--- MATRIZ TRANSACCIONAL ---")
display(df_transacciones)

--- MATRIZ TRANSACCIONAL ---


,"""XY""",$OHO BANI,$uicideboy$,&ME,(G)I-DLE,*NSYNC,.diedlonely,10CM,13 Organisï¿,1da Banton,...,ýýýýýýýýýý,ýýýýýýýýýý ýýýýýýýý,ýýýýýýýýýýýý,ýýýýýýýýýýýý ýýýýýýýýýýýýýý,ýýýýýýýýýýýý ýýýýýýýýýýýýýýýýýýýýýýýýýýý,ýýýýýýýýýýýýýýý ýýýýýýýýýýýýýýý,ýýýýýýýýýýýýýýý(ýýýýýýýýýýýýýýý),ýýýýýýýýýýýýýýýýýý,ýýýýýýýýýýýýýýýýýýýýý ýýýýýýýýýýýýýýýýýýýýý,ýýýýýýýýýýýýýýýýýýýýýýýý
0,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
1,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
2,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
3,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
4,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4590,False,False,True,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
4591,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
4592,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
4593,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False


### Fase 2: Aplicando Apriori
Ahora que tenemos la matriz con ceros y unos, extraigamos los conjuntos de artistas que colaboran frecuentemente.
Debido a que hay miles de artistas solistas, las colaboraciones son estadísticamente raras. Usaremos un `min_support` muy bajo para capturar las colaboraciones.

In [10]:
# --- RETO PARA EL ALUMNO AQUÍ ---

# 1. Encuentra los Itemsets Frecuentes
# Con 0.005 (0.5%) no sale ninguna pareja, ya que la colaboración que más se repite (21 Savage y Metro Boomin)
# solo aparece en 4 de 4595 canciones, por lo que se bajó a 0.0004 (mínimo 2 canciones juntas)
# max_len=2 limita a parejas de artistas para no saturar la memoria RAM
itemsets_frecuentes = apriori(df_transacciones, min_support=0.0004, use_colnames=True, low_memory=True, max_len=2)

# 2. Genera las Reglas de Asociación filtrando por la métrica 'lift' con un mínimo de 1.2
reglas_musicales = association_rules(itemsets_frecuentes, metric='lift', min_threshold=1.2)

# 3. Muestra las reglas resultantes ordenadas por 'confidence' de mayor a menor
display(reglas_musicales.sort_values(by='confidence', ascending=False))

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski
10,(Aqua),(Nicki Minaj),0.000435,0.004788,0.000435,1.000000,208.863636,1.0,0.000433,inf,0.995646,0.090909,1.000000,0.545455
16,(Bryant Myers),(Dei V),0.000653,0.001088,0.000653,1.000000,919.000000,1.0,0.000652,inf,0.999564,0.600000,1.000000,0.800000
41,(L.V.),(Coolio),0.000435,0.000653,0.000435,1.000000,1531.666667,1.0,0.000435,inf,0.999782,0.666667,1.000000,0.833333
118,(TitoM),(S.N.E),0.000435,0.000435,0.000435,1.000000,2297.500000,1.0,0.000435,inf,1.000000,1.000000,1.000000,1.000000
113,(Norma Jean Martine),(Ofenbach),0.000435,0.000653,0.000435,1.000000,1531.666667,1.0,0.000435,inf,0.999782,0.666667,1.000000,0.833333
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
64,(Drake),(SZA),0.018498,0.004788,0.000653,0.035294,7.371658,1.0,0.000564,1.031622,0.880636,0.028846,0.030653,0.085829
13,(Bad Bunny),(Drake),0.013928,0.018498,0.000435,0.031250,1.689338,1.0,0.000178,1.013163,0.413816,0.013605,0.012992,0.027390
12,(Drake),(Bad Bunny),0.018498,0.013928,0.000435,0.023529,1.689338,1.0,0.000178,1.009833,0.415743,0.013605,0.009737,0.027390
58,(Drake),(J. Cole),0.018498,0.003482,0.000435,0.023529,6.757353,1.0,0.000371,1.020530,0.868071,0.020202,0.020117,0.074265


### 🧠 Fase 3: Interpretación de Negocio (El Entregable)

Como Científico de Datos, no basta con entregar una tabla. Debes explicarle los hallazgos al Director Creativo. Revisa tu DataFrame `reglas_musicales` y responde las siguientes 4 preguntas:

**1. Análisis de Soporte (El Volumen):**
Menciona a los dos artistas que conforman la regla con el mayor **Soporte**. ¿Qué significa este número en términos simples para el negocio?

> **Tu Respuesta:** La regla con mayor soporte es la de 21 Savage y Metro Boomin con un soporte de 0.00087, esto quiere decir que aparecen juntos en 4 canciones de 4595, osea en el 0.087% del top de canciones mas escuchadas. Para el negocio significa que es la colaboracion que mas se repite en el dataset, aunque el numero se ve bajo es porque la mayoria de las canciones son de un solo artista, por lo que repetir una pareja 4 veces ya es bastante dentro de los exitos.

**2. Análisis de Confianza (La Dirección):**
Busca una regla que tenga una **Confianza** mayor al 0.5 (50%). Traduce esta regla matemática a una frase que el director pueda entender (Ej. "Si contratamos al artista A, existe un X% de probabilidad de que el artista B también deba estar en el track para que sea el éxito que registramos").

> **Tu Respuesta:** La regla PARTYNEXTDOOR -> Drake tiene una confianza de 0.75. Por lo que se podria interpretar si contratamos a PARTYNEXTDOOR, existe un 75% de probabilidad de que Drake tambien deba estar en el track para que sea el exito que registramos. Esto tiene sentido porque de las 4 canciones de PARTYNEXTDOOR que estan en el top, en 3 aparece Drake, por lo que casi siempre que pega lo hace junto a el.

**3. Análisis de Lift (El Impulso):**
Busca la regla con el **Lift** más alto de toda tu tabla. 
¿Por qué esta métrica es más confiable que el Soporte para decidir si estos dos artistas *realmente* tienen química musical o si solo fue una coincidencia estadística?

> **Tu Respuesta:** La regla con el lift mas alto es la de TitoM y S.N.E con un lift de 2297.5. El lift es mas confiable que el soporte porque compara que tanto aparecen juntos contra lo que se esperaria si fuera solo al azar, el soporte solo dice que tan seguido salen juntos pero no toma en cuenta que un artista puede salir mucho solo por ser popular, como Drake que tiene muchas canciones con muchos artistas diferentes. En este caso un lift tan alto significa que estos dos artistas nunca salen en el top sin el otro, aunque hay que tener cuidado porque solo son 2 canciones, por lo que es mejor revisarlo junto con el soporte.

**4. Toma de Decisión:**
Basado puramente en las reglas matemáticas que descubriste, si la disquera solo tiene presupuesto para juntar a dos artistas en el estudio la próxima semana, ¿qué pareja elegirías y por qué? (Justifica usando las 3 métricas de oro).

> **Tu Respuesta:** Elegiria a Bryant Myers y Dei V, ya que es la pareja que mejor balancea las tres metricas. Tiene un soporte de 0.00065 (3 canciones juntas) que es de los mas altos de toda la tabla, una confianza del 100% en Bryant Myers -> Dei V, osea que todas las canciones de Bryant Myers que estan en el top son con Dei V, y un lift de 919, lo que demuestra que no es coincidencia. Aunque 21 Savage y Metro Boomin tienen mas soporte, su confianza es baja (25% y 13%) porque los dos colaboran con muchos otros artistas, por lo que no seria tan seguro que juntarlos de un exito.